# Bygge applikasjoner for bildegenerering (Azure OpenAI)

Det er mer ved LLM-er enn tekstgenerering. Du kan også generere bilder fra tekstbeskrivelser. Bilder som modalitet er nyttig på tvers av MedTech, arkitektur, turisme, spillutvikling, markedsføring og mer. I denne leksjonen jobber vi med dagens **GPT Image** modeller på Microsoft Foundry.

## Læringsmål

På slutten av denne leksjonen vil du kunne:

- Forklare hva bildegenerering er og hvor det er nyttig.
- Forstå `gpt-image` modellslekten og hvordan den skiller seg fra de eldre DALL·E-modellene.
- Bygge en applikasjon for bildegenerering og redigere bilder.

## Hva er bildegenerering?

Modeller for bildegenerering lager bilder fra en tekstprompt. Moderne modeller som `gpt-image` lærer forholdet mellom tekst og bilder under treningen, og transformerer deretter iterativt tilfeldig støy til et bilde som samsvarer med prompten din.

To kjente familier av bildemodeller er:

- **`gpt-image` (OpenAI)** — nåværende generasjon brukt i denne leksjonen. Den støtter tekst-til-bilde generering og bilderedigering (inpainting med maske).
- **Midjourney** — en populær tredjepartsmodell med egen tjeneste og arbeidsflyt basert på Discord.

> De eldre OpenAI bildemodellene — **DALL·E 2** og **DALL·E 3** — er legacy. DALL·E 3 er ikke lenger tilgjengelig for nye utrullinger, og funksjoner som `create_variation` fantes kun i DALL·E 2. Bruk `gpt-image` modellene for nye applikasjoner.

På Microsoft Foundry er **`gpt-image-2`** den nyeste og mest kapable bildemodellen og den anbefalte standarden. `gpt-image-1.5` og `gpt-image-1-mini` er også generelt tilgjengelige.

> **Viktig:** `gpt-image` modeller returnerer det genererte bildet som **base64** (`b64_json`), ikke som en URL. Koden din dekoder base64-strengen til bytes og lagrer den — det finnes ingen bildelenke å laste ned.


## Lage din første bildeopprettingsapplikasjon

Så hva kreves for å lage en bildeopprettingsapplikasjon? Du trenger følgende biblioteker:

- **python-dotenv**, det anbefales sterkt at du bruker dette biblioteket for å holde hemmelighetene dine i en *.env*-fil adskilt fra koden.
- **openai**, dette biblioteket bruker du for å kommunisere med OpenAI API.
- **pillow**, for å arbeide med bilder i Python.

Hvis du ikke allerede har gjort det, følg instruksjonene på [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst)-siden for å opprette en Microsoft Foundry-ressurs og modell. Velg **gpt-image-2** som modellen (den nyeste Azure OpenAI bildemodellen; DALL·E er eldre).

1. Lag en fil *.env* med følgende innhold:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Finn denne informasjonen i Microsoft Foundry-portalen for ressursen din under seksjonen "Deployments".



1. Samle de ovennevnte bibliotekene i en fil kalt *requirements.txt* slik:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Deretter, opprett et virtuelt miljø og installer bibliotekene:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> For Windows, bruk følgende kommandoer for å opprette og aktivere ditt virtuelle miljø:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Legg til følgende kode i filen kalt *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # konfigurer Azure OpenAI (Microsoft Foundry) klient.
    # Bildemodeller trenger en nylig API-versjon — sjekk Foundry-dokumentasjonen for hvilken versjon din modell krever.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Lag et bilde ved å bruke bildegenererings-API-et
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Skriv inn prompt-teksten her
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # for eksempel "gpt-image-2"
        )
        # Sett katalogen for det lagrede bildet
        image_dir = os.path.join(os.curdir, 'images')

        # Hvis katalogen ikke finnes, opprett den
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialiser bildebane (merk at filtypen bør være png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image-modeller returnerer bildet som base64 (b64_json), ikke en URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Vis bildet i standard bildeviser
        image = Image.open(image_path)
        image.show()

    # fang unntak
    except BadRequestError as err:
        print(err)

    ```

La oss forklare denne koden:

- Først importerer vi bibliotekene vi trenger, inkludert OpenAI-biblioteket, dotenv-biblioteket, base64-modulen og Pillow-biblioteket.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Deretter laster vi miljøvariablene fra *.env*-filen.

    ```python
    # importer dotenv
    dotenv.load_dotenv()
    ```

- Etter det konfigurerer vi Azure OpenAI (Microsoft Foundry) klienten.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Videre genererer vi bildet:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Skriv inn tekst for forespørselen din her
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` modeller returnerer bildet som en **base64** streng i `data[0].b64_json`. Vi dekoder det til bytes og skriver det til en fil — det finnes ingen URL for nedlasting.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Til slutt åpner vi bildet og bruker standard bildevisningsprogram for å vise det:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Flere detaljer om generering av bildet

La oss se på parameterne til `images.generate`:

- **prompt**, er tekstprompten som brukes til å generere bildet. Her er det "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, er størrelsen på det genererte bildet (for eksempel `1024x1024`, `1536x1024`, `1024x1536`, eller `"auto"`).
- **n**, er antall genererte bilder. Her genererer vi ett.
- **model**, er navnet på ditt bildemodellutplassering (for eksempel `gpt-image-2`).

> Bildemodeller tar ikke en `temperature`-parameter — det er en tekstgenereringskontroll. For å få variasjon, kall API-en igjen; for å redusere variasjon, gjør prompten mer spesifikk.

## Ytterligere muligheter ved bildgenerering

Du har sett hvordan man genererer et bilde med noen få linjer Python. `gpt-image` modeller kan også **redigere** et eksisterende bilde. Ved å gi et bilde, en valgfri **maske** (som markerer området som skal endres), og en prompt, kan du endre en del av et bilde — for eksempel legge til en hatt på kaninen vår.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# redigeringer returneres også som base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Grunnbildet inneholder bare kaninen; sluttbildet har hatten på kaninen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
